In [ ]:
library(Seurat)
library(ggplot2)
library(MetaNeighbor)
library(SingleCellExperiment)
library(dplyr)

In [ ]:
hm <- readRDS('data/processed/cortex_analysis_file')
DefaultAssay(hm) <- 'RNA'
hm@meta.data$species <- 'human'
hm

In [ ]:
dp <- readRDS('data/processed/cortex_analysis_file')
DefaultAssay(dp) <- 'RNA'
dp@meta.data$species <- 'Cetacea'
dp

In [ ]:
Idents(hm) <- hm@meta.data$cellType_broad_hc
Idents(dp) <- dp@meta.data$annotation

In [ ]:
selected_hm <- c("Astro", "EndoMural", "Oligo", "OPC", "Inhib", "Excit")
hm <- subset(hm, subset = cellType_broad_hc %in% selected_hm)
selected_dp <- c("Astro", "Endo", "Oligo", "Opc", "In", "Ex")
dp <- subset(dp, subset = annotation %in% selected_dp)
# 统一类别名称
hm@meta.data$cellType_broad_hc <- gsub("EndoMural", "Endo", hm@meta.data$cellType_broad_hc)
hm@meta.data$cellType_broad_hc <- gsub("OPC", "Opc", hm@meta.data$cellType_broad_hc)
hm@meta.data$cellType_broad_hc <- gsub("Inhib", "In", hm@meta.data$cellType_broad_hc)
hm@meta.data$cellType_broad_hc <- gsub("Excit", "Ex", hm@meta.data$cellType_broad_hc)

In [ ]:
table(dp@meta.data$annotation)

In [ ]:
table(hm@meta.data$cellType_broad_hc)

In [ ]:
hm_proportions <- table(hm@meta.data$cellType_broad_hc) / sum(table(hm@meta.data$cellType_broad_hc))
dp_proportions <- table(dp@meta.data$annotation) / sum(table(dp@meta.data$annotation))

hm_df <- data.frame(CellType = names(hm_proportions), Proportion = as.numeric(hm_proportions), Species = "Human")
dp_df <- data.frame(CellType = names(dp_proportions), Proportion = as.numeric(dp_proportions), Species = "Cetacea")

combined_df <- bind_rows(hm_df, dp_df)
print(combined_df)

In [ ]:
p <- ggplot(combined_df, aes(x = CellType, y = Proportion, fill = Species)) +
  geom_bar(stat = "identity", position = position_dodge(width = 0.9), width = 0.8) + # 调整width参数
  geom_text(aes(label = sprintf("%.2f", Proportion)),
            position = position_dodge(width = 0.8), # 与geom_bar相同的对齐方式
            vjust = -0.3, # 垂直对齐
            size = 3) + # 根据需要调整文本大小
  scale_fill_manual(values = c("Human" = "red", "Cetacea" = "blue")) +
  labs(title = "Proportion of Cell Types in Human and Cetacea",
       x = "Cell Type",
       y = "Proportion") +
  theme_minimal() +
  theme(axis.text.x = element_text(angle = 45, hjust = 1))

print(p)
ggsave("results/figures/figure_panel_file", plot = p, width = 10, height = 6, dpi = 300)
ggsave("results/figures/figure_panel_file", plot = p, width = 10, height = 6)

In [ ]:
# 提取表达矩阵和元数据
exprs_hm <- GetAssayData(hm)
exprs_dp <- GetAssayData(dp)

meta_hm <- hm@meta.data
meta_dp <- dp@meta.data

# 提取共有基因
#common_genes <- Reduce(intersect, list(rownames(exprs_hm),rownames(exprs_hm2), rownames(exprs_dp), rownames(exprs_mu)))
common_genes <- Reduce(intersect, list(rownames(exprs_hm),rownames(exprs_dp)))

In [ ]:
length(common_genes)

In [ ]:
# 提取共有基因表达矩阵
exprs_hm_common <- exprs_hm[common_genes, ]
exprs_dp_common <- exprs_dp[common_genes, ]

In [ ]:
# 合并表达矩阵
#combined_exprs <- cbind(exprs_hm_common, exprs_hm2_common, exprs_dp_common, exprs_mu_common)
combined_exprs <- cbind(exprs_hm_common, exprs_dp_common)

In [ ]:
new_colData <- data.frame(
  study_id = c(rep('human', ncol(exprs_hm_common)),rep('dolphin', ncol(exprs_dp_common))),
  cell_type = c(Idents(hm), Idents(dp))
)

# 创建 SingleCellExperiment 对象
sce_combined <- SingleCellExperiment(assays = list(RNA = combined_exprs), colData = new_colData)
sce_combined

In [ ]:
table(sce_combined$study_id)

In [ ]:
table(sce_combined$cell_type)

In [ ]:
var_genes <- variableGenes(dat = sce_combined, exp_labels = sce_combined$study_id)
var_genes

In [ ]:
celltype_NV <- MetaNeighborUS(var_genes = var_genes,
                              dat = sce_combined,
                              study_id = sce_combined$study_id,
                              cell_type = sce_combined$cell_type,
                              fast_version = TRUE)

In [ ]:
png("heatmap_hm2dp_subset.png",width = 1200, height = 1200)
plotHeatmapPretrained(aurocs = celltype_NV, cex = 2, margins = c(22, 22))
dev.off()

In [ ]:
pdf("heatmap_hm2dp_subset.pdf", width = 12, height = 12)
plotHeatmapPretrained(aurocs = celltype_NV, cex = 2, margins = c(20, 20))
dev.off()